# Experiment: Cross-Method Tiny-p Study

Objective:
- Compare `iid`, `mcmc_is`, and `samc` on fixed exact scenarios.
- Track estimates and diagnostics at intermediate estimation points, not just at the final budget.

In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    build_beta_initialization,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    run_named_mcmc_checkpoint_study,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
)

pd.set_option("display.max_columns", 100)
project_root

## Configuration

`ESTIMATION_POINTS` controls the intermediate checkpoints.  
The largest checkpoint is used for the main boxplots; all checkpoints are used for convergence diagnostics.  
In the cross-method notebook these are total budgets. For MCMC-IS, a fixed beta-selection budget is deducted first, and the production chain uses the remaining budget.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "cross_method_notebook"

SCENARIO_GROUP = "core_claim"
SCENARIO_KEYS_OVERRIDE = [
    "gwas_additive_score_ultra_n60",
    "poisson_diffmeans_hep_ultra_n200",
    "gwas_additive_score_sig_n60",
    "poisson_diffmeans_hep_sig_n200",
    "gwas_additive_score_above_n60",
    "poisson_diffmeans_hep_above_n200",
]

ESTIMATION_POINTS = (5_000_000,) if not FAST_MODE else (200_000,)
N_REPEATS = 5 if not FAST_MODE else 2
N_JOBS = min(6, N_REPEATS, os.cpu_count() or 1)
MIN_TAIL_STATES = 2
BASE_SEED = 12_345
MCMC_LOCAL_SCAN_STRATEGY = "adaptive_q"
MCMC_LOCAL_SCAN_OBJECTIVE = "varhat_qmatch_soft"
MCMC_PRODUCTION_ESTIMATOR_VARIANT = "selected_scan_plus_production"
MCMC_SCAN_REFINE_TO_SCREEN_RATIO = 1.0
MCMC_SCAN_FINAL_TO_SCREEN_RATIO = 2.0
MCMC_SCAN_FINALIST_COUNT = 4
DEFAULT_PROPOSAL_SIZE_BY_SAMPLE_BAND = {
    "small": 1,
    "medium": 1,
    "large": 2,
}
MCMC_LOCAL_SCAN_Q_MULTIPLIERS = (0.001, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.33, 0.5, 1.0)
MCMC_LOCAL_SCAN_COARSE_Q_MULTIPLIERS = (0.001, 0.01, 0.10, 0.5)

cross_cfg = CrossMethodStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=N_REPEATS,
    base_seed=BASE_SEED,
    iid_density_samples=150_000 if not FAST_MODE else 10_000,
    min_tail_states=MIN_TAIL_STATES,
    n_jobs=N_JOBS,
)
base_mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=25_000 if not FAST_MODE else 1_000,
    tune_steps=3_000 if not FAST_MODE else 1_000,
    local_scan_screen_total_steps=14_000 if not FAST_MODE else 1_000,
    local_scan_refine_total_steps=14_000 if not FAST_MODE else 1_000,
    local_scan_total_steps=32_000 if not FAST_MODE else 6_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    production_estimator_variant=MCMC_PRODUCTION_ESTIMATOR_VARIANT,
    proposal_size=1,
    local_scan_strategy=MCMC_LOCAL_SCAN_STRATEGY,
    local_scan_objective=MCMC_LOCAL_SCAN_OBJECTIVE,
    local_scan_q_multipliers=MCMC_LOCAL_SCAN_Q_MULTIPLIERS,
    local_scan_coarse_q_multipliers=MCMC_LOCAL_SCAN_COARSE_Q_MULTIPLIERS,
    local_scan_refine_top_k=2,
    local_scan_refine_radius=1,
    local_scan_refine_max_q_points=6,
    local_scan_finalist_count=MCMC_SCAN_FINALIST_COUNT,
)
samc_cfg = SAMCWorkflowConfig(
    n_bins=100,
    t0=1_000.0,
    trace_every=200 if not FAST_MODE else 50,
    lambda_min_pilot=10_000 if not FAST_MODE else 500,
    proposal_size=0.1,
)

def proposal_size_for_scenario(scenario):
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key == "gwas_threshold_suite":
        return 2
    if setting_key == "hep_threshold_suite":
        return 6
    band = str(scenario.portfolio.get("sample_size_band", "medium"))
    return int(DEFAULT_PROPOSAL_SIZE_BY_SAMPLE_BAND.get(band, 1))


def mcmc_proposal_size_for_scenario(scenario):
    return int(proposal_size_for_scenario(scenario))


def samc_cfg_for_scenario(scenario):
    return replace(
        samc_cfg,
        proposal_size=int(proposal_size_for_scenario(scenario)),
    )


def reference_p0_for_scenario(scenario) -> float:
    threshold = scenario.extra.get("known_significance_threshold")
    if threshold is not None:
        threshold_f = float(threshold)
        if threshold_f == threshold_f and threshold_f > 0.0:
            return threshold_f
    return float(scenario.exact_p)


def target_scan_budget_from_p0(p0: float) -> int:
    return 1_000_000


def stage_eval_total(total_steps: int, *, n_candidates: int, n_chains: int) -> int:
    steps_per_chain = _steps_per_chain(int(total_steps), int(n_chains))
    return int(n_candidates) * _mcmc_eval_count(steps_per_chain, int(n_chains))


def adaptive_candidate_counts(cfg: MCMCWorkflowConfig) -> tuple[int, int]:
    coarse_count = len(tuple(cfg.local_scan_coarse_q_multipliers)) or len(tuple(cfg.local_scan_q_multipliers))
    refine_count = int(cfg.local_scan_refine_max_q_points) if str(cfg.local_scan_strategy) == "adaptive_q" else 0
    return int(coarse_count), int(refine_count)


def split_scan_budget_for_scenario(scenario, cfg: MCMCWorkflowConfig) -> dict:
    target_beta_selection_budget = int(target_scan_budget_from_p0(reference_p0_for_scenario(scenario)))
    scan_budget_ex_pilot = max(int(target_beta_selection_budget) - int(cfg.pilot_samples), 1)
    coarse_count, refine_count = adaptive_candidate_counts(cfg)
    finalist_count = int(cfg.local_scan_finalist_count)
    refine_ratio = float(MCMC_SCAN_REFINE_TO_SCREEN_RATIO) if str(cfg.local_scan_strategy) == "adaptive_q" else 0.0
    final_ratio = float(MCMC_SCAN_FINAL_TO_SCREEN_RATIO)
    budget_units = float(coarse_count) + float(refine_count) * refine_ratio + float(finalist_count) * final_ratio
    screen_total_steps = max(int(scan_budget_ex_pilot / max(budget_units, 1.0)), 1)
    refine_total_steps = (
        int(max(1, round(refine_ratio * screen_total_steps)))
        if str(cfg.local_scan_strategy) == "adaptive_q"
        else None
    )
    final_total_steps = int(max(1, round(final_ratio * screen_total_steps)))
    expected_screen_eval_total = stage_eval_total(
        screen_total_steps,
        n_candidates=coarse_count,
        n_chains=int(cfg.local_scan_screen_chains),
    )
    expected_refine_eval_total = (
        stage_eval_total(
            int(refine_total_steps),
            n_candidates=refine_count,
            n_chains=int(
                cfg.local_scan_refine_chains
                if cfg.local_scan_refine_chains is not None
                else cfg.local_scan_screen_chains
            ),
        )
        if refine_total_steps is not None and refine_count > 0
        else 0
    )
    expected_final_eval_total = stage_eval_total(
        final_total_steps,
        n_candidates=finalist_count,
        n_chains=int(cfg.local_scan_chains),
    )
    return {
        "target_beta_selection_budget": int(target_beta_selection_budget),
        "screen_total_steps": int(screen_total_steps),
        "refine_total_steps": int(refine_total_steps) if refine_total_steps is not None else None,
        "final_total_steps": int(final_total_steps),
        "expected_beta_selection_budget_upper": int(
            int(cfg.pilot_samples)
            + int(expected_screen_eval_total)
            + int(expected_refine_eval_total)
            + int(expected_final_eval_total)
        ),
    }


def mcmc_cfg_for_scenario(scenario):
    proposal_size = int(mcmc_proposal_size_for_scenario(scenario))
    scan_budget = split_scan_budget_for_scenario(scenario, base_mcmc_cfg)
    reference_p0 = float(reference_p0_for_scenario(scenario))
    return replace(
        base_mcmc_cfg,
        use_true_p0_for_q_target=False,
        p0_guess=reference_p0,
        proposal_size=proposal_size,
        local_scan_swap_counts=(proposal_size,),
        local_scan_screen_total_steps=int(scan_budget["screen_total_steps"]),
        local_scan_refine_total_steps=(
            int(scan_budget["refine_total_steps"])
            if scan_budget["refine_total_steps"] is not None
            else None
        ),
        local_scan_total_steps=int(scan_budget["final_total_steps"]),
    )

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_GROUP": SCENARIO_GROUP,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "N_REPEATS": N_REPEATS,
    "N_JOBS": N_JOBS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
    "MCMC_LOCAL_SCAN_STRATEGY": MCMC_LOCAL_SCAN_STRATEGY,
    "MCMC_LOCAL_SCAN_OBJECTIVE": MCMC_LOCAL_SCAN_OBJECTIVE,
    "MCMC_PRODUCTION_ESTIMATOR_VARIANT": MCMC_PRODUCTION_ESTIMATOR_VARIANT,
    "MCMC_SCAN_FINALIST_COUNT": MCMC_SCAN_FINALIST_COUNT,
    "MCMC_LOCAL_SCAN_Q_MULTIPLIERS": MCMC_LOCAL_SCAN_Q_MULTIPLIERS,
    "MCMC_LOCAL_SCAN_COARSE_Q_MULTIPLIERS": MCMC_LOCAL_SCAN_COARSE_Q_MULTIPLIERS,
        "DEFAULT_PROPOSAL_SIZE_BY_SAMPLE_BAND": DEFAULT_PROPOSAL_SIZE_BY_SAMPLE_BAND,
        "APPLICATION_PROPOSAL_SIZE_OVERRIDES": {
            "gwas_threshold_suite": 2,
            "hep_threshold_suite": 6,
        },
        "REFERENCE_P0_MODE": "known_significance_threshold_else_exact_p",
    }, indent=2))

## Load Scenarios

In [ ]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None if SCENARIO_KEYS_OVERRIDE is not None else SCENARIO_GROUP,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "cross_method") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
        "family": s.portfolio.get("family"),
        "rarity_band": s.portfolio.get("rarity_band"),
        "difficulty": s.portfolio.get("expected_difficulty"),
        "groups": ",".join(s.portfolio.get("groups", [])),
    }
    for s in scenarios
])

## Run Cross-Method Study

For each scenario:
- build one MCMC-IS beta workflow,
- evaluate all methods at every checkpoint in `ESTIMATION_POINTS`,
- save max-budget and convergence plots.

In [ ]:
cross_results = {}

for scenario in scenarios:
    scenario_mcmc_cfg = mcmc_cfg_for_scenario(scenario)
    scenario_samc_cfg = samc_cfg_for_scenario(scenario)
    print(f"Running {scenario.key} | exact p={scenario.exact_p:.3e}")
    print(json.dumps({
        "scenario": scenario.key,
        "application_setting_key": scenario.extra.get("application_setting_key"),
        "known_significance_threshold": scenario.extra.get("known_significance_threshold"),
        "reference_p0_for_qtarget": reference_p0_for_scenario(scenario),
        "sample_size_band": scenario.portfolio.get("sample_size_band"),
        "mcmc_proposal_size": scenario_mcmc_cfg.proposal_size,
        "samc_proposal_size": scenario_samc_cfg.proposal_size,
        "mcmc_local_scan_strategy": scenario_mcmc_cfg.local_scan_strategy,
        "mcmc_local_scan_swap_counts": scenario_mcmc_cfg.local_scan_swap_counts,
        "mcmc_local_scan_objective": scenario_mcmc_cfg.local_scan_objective,
        "mcmc_production_estimator_variant": scenario_mcmc_cfg.production_estimator_variant,
        "target_beta_selection_budget": target_scan_budget_from_p0(reference_p0_for_scenario(scenario)),
        "local_scan_screen_total_steps": scenario_mcmc_cfg.local_scan_screen_total_steps,
        "local_scan_refine_total_steps": scenario_mcmc_cfg.local_scan_refine_total_steps,
        "local_scan_final_total_steps": scenario_mcmc_cfg.local_scan_total_steps,
    }, indent=2))
    study = run_cross_method_study(
        scenario,
        cross_cfg=cross_cfg,
        mcmc_cfg=scenario_mcmc_cfg,
        samc_cfg=scenario_samc_cfg,
    )
    cross_results[scenario.key] = study

    if SAVE_OUTPUTS and run_dir is not None:
        save_cross_method_outputs(
            scenario,
            study,
            output_dir=run_dir / scenario.key,
            cross_cfg=cross_cfg,
            mcmc_cfg=scenario_mcmc_cfg,
            samc_cfg=scenario_samc_cfg,
        )

    print(json.dumps({
        "scenario": scenario.key,
        "mcmc_beta_selection_budget": study["mcmc_beta_selection_budget"],
        "mcmc_reported_checkpoints": study.get("mcmc_reported_checkpoints", []),
        "beta_used": study["beta_workflow"]["beta_used"],
    }, indent=2))
    summary_df = pd.DataFrame(study["summary"]).sort_values(["checkpoint", "method"])
    display(summary_df[[
        "checkpoint",
        "method",
        "mean_estimate",
        "rmse",
        "mean_variance_estimate",
        "mean_eval_incl_tuning",
        "mean_q_tilt_tail_share",
        "mean_ess",
        "mean_zero_rate",
        "mean_samc_max_rel_freq_error",
    ]])

family_rows = []
for scenario in scenarios:
    meta = scenario.portfolio
    for row in cross_results[scenario.key]["summary"]:
        family_rows.append({
            "scenario": scenario.key,
            "family": meta.get("family"),
            "rarity_band": meta.get("rarity_band"),
            "difficulty": meta.get("expected_difficulty"),
            **row,
        })

family_df = pd.DataFrame(family_rows)
display(
    family_df.groupby(["family", "rarity_band", "method", "checkpoint"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_estimate=("mean_estimate", "mean"),
        mean_q_tilt_tail_share=("mean_q_tilt_tail_share", "mean"),
        mean_ess=("mean_ess", "mean"),
    )
    .sort_values(["family", "rarity_band", "checkpoint", "method"])
)

## Review Saved Figures

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    for scenario in scenarios:
        scenario_dir = run_dir / scenario.key
        print(f"\n{scenario.key}")
        display(Image(filename=str(scenario_dir / "cross_method_max_budget.png")))
        display(Image(filename=str(scenario_dir / "cross_method_convergence.png")))
        display(Image(filename=str(scenario_dir / "cross_method_diagnostics.png")))
        display(Image(filename=str(scenario_dir / "iid_density.png")))
else:
    print("SAVE_OUTPUTS=False, so no saved figures to display.")

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_SCENARIO_DIR = None
# # Example:
# # RELOAD_SCENARIO_DIR = project_root / "results" / "cross_method_notebook" / "20260306_120000_cross_method" / "gwas_additive_score_sig_n60"

# if RELOAD_SCENARIO_DIR is not None:
#     saved = load_cross_method_saved_output(RELOAD_SCENARIO_DIR)
#     print(json.dumps({
#         "scenario": saved["metadata"]["scenario"],
#         "exact_p": saved["metadata"]["exact_p"],
#         "mcmc_beta_selection_budget": saved["metadata"]["beta_workflow"]["beta_selection_eval_total"],
#     }, indent=2))
#     regen = regenerate_cross_method_plots_from_saved(RELOAD_SCENARIO_DIR)
#     for name, path in regen.items():
#         print(name, path)
#         display(Image(filename=str(path)))
# else:
#     print("Set RELOAD_SCENARIO_DIR to a saved scenario directory to regenerate plots from disk only.")